# 12. Three classic econometric relationships, from real data

Economists distrust black boxes for a good reason: a coefficient that cannot be
explained to a policymaker is not much use, however well it predicts. `beamfeat`
returns an explicit formula and a q-value, which is closer to what empirical
economics actually wants.

This notebook tests that claim on **three real datasets**, one per classic model:

| Part | Model | Data | Source |
|---|---|---|---|
| 1 | Gravity model | Canadian interprovincial migration, 90 flows | `carData/Migration` |
| 2 | Cobb–Douglas production | US states, 816 state-years | `Ecdat/Produc` |
| 3 | Misery index | US macro, quarterly 1957–2005 | `AER/USMacroSW` |

All three come from the [Rdatasets](https://vincentarelbundock.github.io/Rdatasets/)
archive, downloaded once into `csv/`. Direct links are in each section.

**The recurring lesson.** All three models are multiplicative or additive *in logs*.
Getting the scale right matters more than anything the search does — the same point
notebook 07 makes with Dittus–Boelter.


## Corrections to a widely circulated write-up

A blog post on `beamfeat` for econometrics has been going around. Its economics is
mostly sound, but the code will not run and one formula is wrong. Checked against
`beamfeat 0.1.1`:

| Claim | Reality |
|---|---|
| `from beamfeat import BeamFeat` | No such class. It is `BeamFeatRegressor`, `BeamFeatClassifier`, `BeamFeatTransformer`. |
| `operators=['add','sub','mul','div','log']` | Not a parameter. Use `unary_ops=` and `binary_ops=` separately. |
| `fdr_level=0.05` | Not a parameter. It is `target_fdr=`. |
| `control_method='knockoff'` | Not a parameter. It is `selector='knockoff'` — and see the warning in Part 4. |
| Cobb–Douglas is `log(L) * log(K)` | **Wrong.** Cobb–Douglas is $Y = A L^\alpha K^\beta$, so $\log Y = \log A + \alpha \log L + \beta \log K$ — *additive* in logs, not multiplicative. Part 2 shows why this matters. |
| Knockoffs "pioneered by Barber and Candès in 2014" | arXiv 2014; published *Annals of Statistics* **43**(5), 2015. |

The post's FDP arithmetic is right: knockoff+ thresholds on
$(1 + \#\{W_j \le -t\}) / (\#\{W_j \ge t\} \vee 1)$. Part 4 shows the practical
consequence it omits.


## Setup


In [ ]:
%pip install -q "beamfeat[units]" pandas matplotlib seaborn scikit-learn


## Environment

Run this before anything else. It records the exact stack these results came from, so a
rerun on a different machine is comparable rather than merely similar.

`REQUIRE` lists what *this* notebook needs. The tutorials need only `beamfeat`; a full
benchmark rerun also needs the comparison libraries, and the cell will tell you how to
build that environment if they are absent.


In [ ]:
# --- environment provenance --------------------------------------------------
# Tutorials need only beamfeat. For a full benchmark rerun set:
#   REQUIRE = ["beamfeat", "lightgbm", "autofeat", "openfe", "knockpy"]
REQUIRE = ["beamfeat"]

import importlib.metadata as md
import importlib.util
import os, pathlib, platform, sys

ROOT = pathlib.Path.cwd()
if ROOT.name == "benchmarks":
    ROOT = ROOT.parent
BENCH = ROOT / "benchmarks"
if BENCH.is_dir():                      # only when run from inside the repo
    sys.path.insert(0, str(BENCH))

print(f"python       {sys.version.split()[0]}")
print(f"platform     {platform.platform()}")
print(f"cpu count    {os.cpu_count()}")

for name in ("numpy", "scipy", "scikit-learn", "pandas", "matplotlib", "seaborn",
             "cantera", "pint", "lightgbm", "autofeat", "openfe", "knockpy",
             "beamfeat"):
    try:
        print(f"{name:<13} {md.version(name)}")
    except md.PackageNotFoundError:
        print(f"{name:<13} {'MISSING' if name in REQUIRE else '-'}")

print(f"interpreter  {sys.executable}")

missing = [p for p in REQUIRE if importlib.util.find_spec(p) is None]
if missing:
    raise SystemExit(
        f"missing: {missing}\n"
        "Install them, or build the pinned comparison environment:\n"
        "    conda create -n af315 python=3.11 -y && conda activate af315\n"
        "    bash benchmarks/independent/setup_env.sh")


In [ ]:
import warnings
from pathlib import Path
import urllib.request, urllib.error

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from beamfeat import BeamFeatRegressor, BeamFeatTransformer, KnockoffSelector

warnings.filterwarnings("ignore", message=".*valid feature names.*")
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)

CSV_DIR = Path("csv")
CSV_DIR.mkdir(exist_ok=True)          # created on first run, reused after

RDATASETS = "https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/"


def load_rdataset(package, item):
    # Download csv/<item>.csv from the Rdatasets archive once, then read from disk.
    path = CSV_DIR / f"{item}.csv"
    if path.exists():
        print(f"cached   {path}")
    else:
        url = RDATASETS + f"{package}/{item}.csv"
        print(f"fetching {url} ...", end=" ", flush=True)
        try:
            urllib.request.urlretrieve(url, path)
        except urllib.error.URLError as err:
            raise RuntimeError(f"download failed: {url}\n{err}") from None
        print(f"saved to {path}")
    return pd.read_csv(path)


SEED = 0


---
## Part 1 — The gravity model of migration

Bilateral flows are predicted to scale with the economic mass of both ends and decay
with distance:

$$F_{ij} \;\propto\; \frac{P_i^{\alpha}\,P_j^{\beta}}{D_{ij}^{\gamma}}$$

with $\alpha = \beta = \gamma = 1$ in the textbook form. Originally written for trade;
it works just as well for migration.

**Data:** every flow between Canadian provinces, 1966–71. 90 origin–destination pairs
with populations at both ends and the distance between them.

<https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/carData/Migration.csv>


In [ ]:
mig = load_rdataset("carData", "Migration")
print(mig.shape)
mig.head()


In [ ]:
G = mig[["pops71", "popd71", "distance", "migrants"]].rename(
    columns={"pops71": "pop_origin", "popd71": "pop_dest"})

print("missing:", int(G.isna().sum().sum()))
G.describe().T[["mean", "std", "min", "max"]]


Note the spread: migrants runs from a few hundred to 99,430 — nearly three orders of
magnitude. Any multiplicative relationship on that scale should be modelled in logs.


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))

ax[0].scatter(G.distance, G.migrants, s=25, alpha=.6)
ax[0].set_xlabel("distance (km)"); ax[0].set_ylabel("migrants")
ax[0].set_title("Raw scale — one blob and a tail")

gravity = G.pop_origin * G.pop_dest / G.distance
ax[1].scatter(gravity, G.migrants, s=25, alpha=.6)
ax[1].set_xlabel(r"$P_i P_j / D$"); ax[1].set_ylabel("migrants")
ax[1].set_title(f"Textbook feature (r = {np.corrcoef(gravity, G.migrants)[0,1]:.2f})")

ax[2].scatter(np.log(gravity), np.log(G.migrants), s=25, alpha=.6)
ax[2].set_xlabel(r"$\log(P_i P_j / D)$"); ax[2].set_ylabel("log migrants")
ax[2].set_title("Log-log — the relationship appears")

plt.tight_layout(); plt.show()


The middle panel is the point. Even the *correct* textbook feature looks unconvincing
on the raw scale, because a handful of large flows dominate the squared error. On the
log-log axes the relationship is obvious.


In [ ]:
X_raw = G[["pop_origin", "pop_dest", "distance"]]
y_raw = G.migrants.values

X_log = np.log(X_raw)
y_log = np.log(y_raw)

m_raw = BeamFeatRegressor(max_depth=2, beam_width=40, random_state=SEED).fit(X_raw, y_raw)
m_log = BeamFeatRegressor(max_depth=2, beam_width=40, random_state=SEED).fit(X_log, y_log)

print(f"raw scale : R2 = {m_raw.score(X_raw, y_raw):.4f}   {m_raw.formulas()[:2]}")
print(f"log scale : R2 = {m_log.score(X_log, y_log):.4f}   {m_log.formulas()[:2]}")


Neither returns a clean $P_i P_j / D$, and that is worth understanding rather than
explaining away.

**In logs the gravity model is linear.** Taking logs of both sides gives

$$\log F = c + \alpha \log P_i + \beta \log P_j - \gamma \log D$$

which is ordinary least squares on the three log columns — no construction required.
When you already know the functional form, feature search is the wrong tool. Use it to
*estimate* the elasticities instead.


In [ ]:
ols = LinearRegression().fit(X_log.values, y_log)

print(f"{'':<22}{'estimated':>11}{'textbook':>11}")
print("-" * 44)
for name, coef, textbook in zip(["log pop_origin", "log pop_dest", "log distance"],
                                ols.coef_, [1.0, 1.0, -1.0]):
    print(f"{name:<22}{coef:>+11.3f}{textbook:>+11.1f}")
print(f"\nR2 = {ols.score(X_log.values, y_log):.4f}")


Both population elasticities are positive and the distance elasticity is negative,
each within about 0.25 of the textbook unit values — from 90 observations of provincial
migration, with no theory supplied.

That the estimates fall short of 1.0 is itself standard: distance proxies for language,
provincial policy and network effects, and unit elasticities are a benchmark rather
than a law.


---
## Part 2 — Cobb–Douglas, and the error worth catching

$$Y = A\,L^{\alpha} K^{\beta} \qquad\Longrightarrow\qquad \log Y = \log A + \alpha \log L + \beta \log K$$

**Additive in logs.** The circulating write-up says the discovered feature should be
`log(L) * log(K)`; that is a different function entirely, and it has no elasticity
interpretation.

Under constant returns to scale $\alpha + \beta = 1$, which gives a testable
prediction rather than a curve fit.

**Data:** 48 US states, 1970–86, 816 state-years — gross state product, employment,
private capital and public capital.

<https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/Ecdat/Produc.csv>


In [ ]:
prod = load_rdataset("Ecdat", "Produc")
print(prod.shape, "|", prod.state.nunique(), "states,",
      prod.year.min(), "-", prod.year.max())
prod.head()


In [ ]:
P = prod[["gsp", "emp", "pc", "pcap"]].rename(columns={
    "gsp": "output", "emp": "labour", "pc": "capital", "pcap": "public_capital"})

print("missing:", int(P.isna().sum().sum()))
P.describe().T[["mean", "std", "min", "max"]]


In [ ]:
P_log = np.log(P)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(P_log.corr(), annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, cbar=False, ax=ax[0])
ax[0].set_title("Log variables are highly collinear")

ax[1].scatter(P_log.labour, P_log.output, s=8, alpha=.4)
ax[1].set_xlabel("log labour"); ax[1].set_ylabel("log output")
ax[1].set_title("Log-log — tight and linear")

plt.tight_layout(); plt.show()


Correlations above 0.97 between the log inputs. Keep that in view — it is the
multicollinearity problem the write-up warns about, and it is real here.


In [ ]:
X_cd = P_log[["labour", "capital"]]
y_cd = P_log.output.values

ols_cd = LinearRegression().fit(X_cd.values, y_cd)
alpha, beta = ols_cd.coef_

print(f"labour elasticity  (alpha) : {alpha:.3f}")
print(f"capital elasticity (beta)  : {beta:.3f}")
print(f"alpha + beta               : {alpha + beta:.3f}   (constant returns => 1.000)")
print(f"R2                         : {ols_cd.score(X_cd.values, y_cd):.5f}")


**Returns to scale within five percent of one**, and a labour share near 0.7 — both
squarely in line with the empirical literature, from raw state accounts.

This is the result an economist would actually publish. Note that it came from OLS on
log columns, not from the search.


In [ ]:
# What does the search return in log space?
m_cd = BeamFeatRegressor(max_depth=2, beam_width=40, random_state=SEED).fit(X_cd, y_cd)

print("discovered :", m_cd.formulas()[:3])
print(f"R2         : {m_cd.score(X_cd, y_cd):.5f}   (OLS: {ols_cd.score(X_cd.values, y_cd):.5f})")


The search finds `labour * capital` among its features — which, since the columns are
already logged, is exactly the `log(L) * log(K)` the write-up describes. It fits
marginally better than Cobb–Douglas.

**And it is still the wrong model.** `log(L) * log(K)` has no elasticity
interpretation, does not nest constant returns to scale, and cannot be compared with
sixty years of published estimates. A slightly higher R² is not worth that.

This is the clearest case in the series of the paper's own caveat: **the guarantee
certifies association, not structure.** When theory gives you the functional form,
impose it and estimate the parameters. Use search where theory is silent.


In [ ]:
# Does public capital earn its place? A question theory does not settle.
ols_3 = LinearRegression().fit(P_log[["labour", "capital", "public_capital"]].values, y_cd)

print("with public capital:")
for name, coef in zip(["labour", "capital", "public_capital"], ols_3.coef_):
    print(f"  {name:<16} {coef:+.3f}")
print(f"  sum of the three : {ols_3.coef_.sum():.3f}")
print(f"  R2               : {ols_3.score(P_log[['labour','capital','public_capital']].values, y_cd):.5f}"
      f"   (two-input: {ols_cd.score(X_cd.values, y_cd):.5f})")


Public capital takes a positive coefficient around 0.15 and barely moves R². This is
the Aschauer public-capital debate in miniature — and exactly where a constructor with
error control has something to add, because theory does not tell you the answer.


---
## Part 3 — The misery index

Okun's index is simply

$$\text{Misery} = \text{inflation} + \text{unemployment}$$

A crude sum, and the claim is that it beats either component alone. That is a testable
proposition, not a definition.

**Data:** US macro series, quarterly 1957–2005. We build inflation from CPI ourselves.

<https://raw.githubusercontent.com/vincentarelbundock/Rdatasets/master/csv/AER/USMacroSW.csv>


In [ ]:
macro = load_rdataset("AER", "USMacroSW")

# annualised quarterly CPI inflation; first row is lost to differencing
macro["inflation"] = 400 * np.log(macro.cpi).diff()
macro = macro.dropna().reset_index(drop=True)
macro["misery"] = macro.inflation + macro.unemp

print(f"{len(macro)} quarters")
macro[["unemp", "inflation", "misery", "tbill", "tbond"]].describe().T[
    ["mean", "std", "min", "max"]].round(2)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(macro.index, macro.unemp, label="unemployment")
ax[0].plot(macro.index, macro.inflation, label="inflation")
ax[0].plot(macro.index, macro.misery, lw=2, label="misery = sum")
ax[0].set_xlabel("quarter since 1957"); ax[0].set_ylabel("percent")
ax[0].set_title("The two components and their sum"); ax[0].legend()

ax[1].scatter(macro.misery, macro.tbond, s=18, alpha=.6)
ax[1].set_xlabel("misery index"); ax[1].set_ylabel("10-year yield")
ax[1].set_title("Misery vs long rates")

plt.tight_layout(); plt.show()


The stagflation of the late 1970s is the twin peak — both components high at once,
which is precisely the situation the index was invented to describe.


In [ ]:
target = macro.tbond.values          # 10-year Treasury yield

print("correlation with the 10-year yield")
for name, series in [("unemployment", macro.unemp),
                     ("inflation", macro.inflation),
                     ("misery (the sum)", macro.misery)]:
    print(f"  {name:<20} {np.corrcoef(series, target)[0,1]:+.3f}")


**The sum beats both components.** +0.71 against +0.66 for inflation and +0.34 for
unemployment. The misery index is doing real work, not just adding two numbers.


In [ ]:
X_m = macro[["unemp", "inflation"]]

ols_two = LinearRegression().fit(X_m.values, target)
ols_mis = LinearRegression().fit(macro[["misery"]].values, target)
m_search = BeamFeatRegressor(max_depth=2, beam_width=40, random_state=SEED).fit(X_m, target)

print(f"{'model':<34}{'params':>8}{'R2':>9}")
print("-" * 51)
print(f"{'OLS on both components':<34}{2:>8}{ols_two.score(X_m.values, target):>9.4f}")
print(f"{'OLS on the misery index alone':<34}{1:>8}{ols_mis.score(macro[['misery']].values, target):>9.4f}")
print(f"{'beamfeat constructed features':<34}{len(m_search.formulas()):>8}"
      f"{m_search.score(X_m, target):>9.4f}")
print("\nbeamfeat features:", m_search.formulas()[:3])


Read the parameter column. **One hand-made feature matches a two-variable regression**
— the misery index is an efficient summary, which is the whole claim behind it.

`beamfeat` edges both on R² using products rather than sums. Whether that is worth
having depends entirely on what you need the model for: for forecasting, take it; for
a paper about *misery*, the sum is the object of study and the products are not.


---
## Part 4 — The pipeline economists actually want

Search for the form, then hand the surviving features to OLS so the coefficients stay
interpretable. `BeamFeatTransformer` is the piece that makes this work in a
`scikit-learn` pipeline.


In [ ]:
pipe = Pipeline([
    ("construct", BeamFeatTransformer(max_depth=2, beam_width=30,
                                      random_state=SEED, target_fdr=0.05)),
    ("ols", LinearRegression()),
]).fit(X_cd, y_cd)

constructed = pipe.named_steps["construct"]
print("surviving features :", constructed.formulas())
print("OLS coefficients   :", np.round(pipe.named_steps["ols"].coef_, 4))
print(f"R2                 : {pipe.score(X_cd, y_cd):.5f}")


### The knockoff trap, diagnosed and fixed

The circulating write-up recommends `control_method='knockoff'` with `fdr_level=0.05`.
Beyond the parameter names being wrong, that combination **selects nothing** here — and
the pipeline then fails outright, because OLS cannot fit zero columns.


In [ ]:
try:
    Pipeline([
        ("construct", BeamFeatTransformer(max_depth=2, beam_width=30, random_state=SEED,
                                          target_fdr=0.05, selector="knockoff")),
        ("ols", LinearRegression()),
    ]).fit(X_cd, y_cd)
except ValueError as err:
    print(f"ValueError: {str(err)[:90]}")


#### The arithmetic

Knockoff+ thresholds on

$$\frac{1 + \#\{W_j \le -t\}}{\#\{W_j \ge t\} \vee 1} \;\le\; q$$

That numerator starts at **1** whatever the data. Even with zero features favouring
their knockoffs, satisfying the inequality needs

$$\#\{W_j \ge t\} \;\ge\; 1/q$$

So at $q = 0.05$ **at least 20 features must clear the threshold**, or nothing can be
selected. This is a property of knockoff+, not a defect.


In [ ]:
def knockoff_floor(q):
    # Minimum number of features that must clear the threshold for knockoff+
    # to be able to select anything at level q.
    return int(np.ceil(1.0 / q))


for q in [0.01, 0.05, 0.10, 0.20, 0.30]:
    print(f"  target_fdr = {q:<5}  ->  needs at least {knockoff_floor(q):>3} discoveries")


#### The obvious fix does not work

The natural reaction is "give it a bigger candidate pool". Test that directly — three
input sets, four FDR levels, tracking how many candidates the selector actually screens
through.


In [ ]:
INPUT_SETS = {
    "2 inputs": ["emp", "pc"],
    "3 inputs": ["emp", "pc", "pcap"],
    "7 inputs": ["emp", "pc", "pcap", "hwy", "water", "util", "unemp"],
}

# The selector prints its diagnostic on every fit; it was shown once above, so
# silence it here to keep the table readable.
import logging
logging.getLogger("beamfeat.selection").setLevel(logging.ERROR)

rows = []
for label, cols in INPUT_SETS.items():
    Xi = np.log(prod[cols].clip(lower=1e-9))
    for q in [0.05, 0.10, 0.20, 0.30]:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            tf = BeamFeatTransformer(max_depth=2, beam_width=40, random_state=SEED,
                                     target_fdr=q, selector="knockoff").fit(Xi, y_cd)
        rep = pd.DataFrame(tf.selection_report_)
        rows.append({"inputs": label, "pool": len(rep), "target_fdr": q,
                     "needs": knockoff_floor(q),
                     "screened": int(rep.screened.sum()),
                     "kept": int(rep.kept.sum())})

pd.DataFrame(rows)


Read the `needs` and `screened` columns together.

Going from 2 inputs to 7 grows the pool from 9 candidates to 40 — and knockoff+ at
$q=0.05$ still selects **nothing**. It never gets past $q=0.20$, and only then by
screening 6 or 7 features.

**The binding constraint is the signal, not the pool.** The number of candidates that
genuinely beat their knockoffs is set by how much real structure the data contains.
This production function has a handful of true relationships, so no amount of extra
candidates will manufacture the twenty discoveries $q=0.05$ demands. Enlarging the pool
adds noise candidates, which is the opposite of help.


#### What actually works

The permutation selector — `beamfeat`'s default, and the one the paper recommends for
engineered candidates because they share parents and are correlated by construction —
has no such floor.


In [ ]:
comparison = []
for q in [0.01, 0.05, 0.10, 0.20]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        perm = BeamFeatTransformer(max_depth=2, beam_width=30, random_state=SEED,
                                   target_fdr=q, selector="permutation").fit(X_cd, y_cd)
        knock = BeamFeatTransformer(max_depth=2, beam_width=30, random_state=SEED,
                                    target_fdr=q, selector="knockoff").fit(X_cd, y_cd)
        mod = BeamFeatTransformer(max_depth=2, beam_width=30, random_state=SEED,
                                  target_fdr=q,
                                  selector=KnockoffSelector(offset=0)).fit(X_cd, y_cd)
    comparison.append({
        "target_fdr": q,
        "permutation (default)": len(perm.formulas()),
        "knockoff+ (true FDR)": len(knock.formulas()),
        "knockoff offset=0 (modified FDR)": len(mod.formulas()),
    })

pd.DataFrame(comparison).set_index("target_fdr")


The permutation route returns the same feature at every level from 1% to 20% — the
Cobb–Douglas interaction is strong enough that the FDR level barely matters. That
stability is what you want before quoting a q-value in a paper.

`offset=0` also returns features, but it controls
$\mathbb{E}[V/(R + q^{-1})]$ — a *modified* FDR, not the FDR. If you use it, say so in
writing.


#### A guard worth pasting into your own code

The real problem is not that knockoff+ selected nothing — that was correct. It is that
the failure surfaced as a `ValueError` from `LinearRegression` three steps downstream.
This wrapper fails at the right place with an actionable message.


In [ ]:
def fit_constructor(X, y, target_fdr=0.05, selector="permutation", **kw):
    # Fit a BeamFeatTransformer and refuse to return zero columns silently.
    tf = BeamFeatTransformer(random_state=SEED, target_fdr=target_fdr,
                             selector=selector, **kw).fit(X, y)
    rep = pd.DataFrame(tf.selection_report_)
    info = {"pool": len(rep),
            "screened": int(rep.screened.sum()),
            "kept": int(rep.kept.sum())}

    if info["kept"] == 0:
        name = selector if isinstance(selector, str) else type(selector).__name__
        raise RuntimeError(
            f"no features survived (pool={info['pool']}, selector={name}, "
            f"q={target_fdr}).\n"
            f"knockoff+ cannot select fewer than {knockoff_floor(target_fdr)} features "
            f"at q={target_fdr}. Options: selector='permutation' (the default); "
            f"raise target_fdr above {1/info['pool']:.2f}; or "
            "KnockoffSelector(offset=0) for modified FDR.")
    return tf, info


for selector in ["knockoff", "permutation"]:
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            tf, info = fit_constructor(X_cd, y_cd, target_fdr=0.05, selector=selector,
                                       max_depth=2, beam_width=30)
        model = Pipeline([("c", tf), ("ols", LinearRegression())]).fit(X_cd, y_cd)
        print(f"OK    {selector:<12} {info}  R2 = {model.score(X_cd, y_cd):.5f}")
    except RuntimeError as err:
        print(f"STOP  {selector:<12} {err}\n")


Same underlying behaviour, but the message now names the cause and three ways out
instead of complaining about array shapes.

**The interpretation.** Knockoff+ is the stricter instrument and it declined to certify
anything at 5% on this data — correctly, because two-input Cobb–Douglas simply does not
contain twenty independent discoveries. The permutation route, which does not carry
that structural floor, certified the interaction at every level down to 1%.

Two selectors, two different questions. The failure was informative; the crash was not.


---
## Two pitfalls that matter more in economics than elsewhere

**Confounding.** A q-value of 0.05 says a feature is not noise. It says nothing about
causality. If a major confounder is missing from your columns, the search will build
confident features out of biased data — it cannot know what you failed to measure.
`beamfeat` does not replace a causal identification strategy, and nothing in its output
should be read as an effect.

**Multicollinearity.** Part 2's log inputs correlate above 0.97. If the search returns
both `GDP` and `GDP / population`, feeding both to OLS inflates standard errors and
wrecks your p-values. The parsimony step prunes some of this, but check the correlation
matrix of the *constructed* features before running inference on them.


In [ ]:
# Always check this before quoting standard errors.
Z = pd.DataFrame(np.asarray(constructed.transform(X_cd)),
                 columns=constructed.formulas())
Z["labour"] = X_cd.labour.values
Z["capital"] = X_cd.capital.values

print("correlation among constructed + raw features:")
print(Z.corr().round(3))


## Takeaways

1. **Gravity model** — log-log OLS on 90 migration flows gives population elasticities
   of the right sign and roughly the right size, and a negative distance elasticity.
   In logs the model is linear, so search is not the tool; estimation is.
2. **Cobb–Douglas** — labour and capital elasticities sum to within five percent of
   one, from 816 state-years. The search finds `log(L) * log(K)` and fits marginally
   better; it is still the wrong model, because it has no elasticity interpretation.
3. **Misery index** — the sum correlates with long rates better than either component,
   and one hand-made feature matches a two-variable regression.
4. **All three are linear in the right coordinates.** Choosing the scale mattered more
   than anything the search did.
5. **knockoff+ at `target_fdr=0.05` requires a pool of more than 20 candidates.** On
   small pools it correctly selects nothing. Use the permutation default.
6. Reach for construction where theory is silent — public capital in Part 2 — not
   where it already gives you the form.
